# Evidencias individuales de Idiomas

Construye `data/evidencias/idiomas/evidencias_idiomas.csv` (`IDIOMA`).
Reglas en `README.md` y formato de atributos en `../ATRIBUTOS.md`. Requiere haber generado antes las evidencias de trayectoria.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "02_evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.idiomas import construir_evidencias_idiomas

pd.set_option("display.max_colwidth", 200)

In [ ]:
evidencias, reporte, validacion = construir_evidencias_idiomas(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

## Validación (todas las filas deben tener `n_fallas = 0`)

In [ ]:
validacion

## Ejemplos

In [ ]:
for _, r in evidencias.sample(6, random_state=7).iterrows():
    print(f"- {r.evidencia_id} | {r.texto}")
    print("  ", json.dumps(json.loads(r.atributos), ensure_ascii=False))

## Casos con más de un nivel declarado en el mismo idioma

Registros de la fuente y la evidencia que generan: cada tipo de nivel es una lista con todos los niveles declarados (de mayor a menor) y el texto usa el mayor.

In [ ]:
from evidencias.idiomas.fuentes import cargar_idiomas, cargar_poblacion

attrs = evidencias["atributos"].map(json.loads)
listas = ["niveles_conversacion", "niveles_lectura", "niveles_escritura", "niveles_mcer"]
varios = evidencias[attrs.map(lambda a: any(len(a[k]) > 1 for k in listas))]
print(f"Evidencias con más de un nivel declarado en alguna habilidad: {len(varios)}")

fuente = cargar_idiomas(cargar_poblacion())
cols = ["IDIOMA", "NIVELCONVERSACION", "NIVELLECTURA", "NIVELESCRITURA", "NIVELCOMPRESION", "NIVELMCER", "LENGUANATIVA"]
for _, r in varios.head(5).iterrows():
    a = json.loads(r["atributos"])
    print()
    print("=" * 90)
    print("Registros en la fuente (A=avanzado, I=intermedio, B=básico, N=ninguna):")
    print(fuente[fuente["IDIDIOMAPERSONA"].isin(a["_origen"]["ids_idioma_persona"])][cols].to_string(index=False))
    print()
    print("Texto:", r["texto"])
    for k in ["idioma", *listas, "niveles_comprension", "es_lengua_nativa"]:
        print(f"  {k}: {a[k]}")

## Registros no considerados (para revisión)

In [ ]:
pd.read_csv(PROJECT_ROOT / "data" / "evidencias" / "idiomas" / "registros_no_considerados.csv")["motivo"].value_counts()